# AVA-VLA LIBERO+ paired rollout

Compare T0, T1 (1975), and suite-balanced FFT (725) on Spatial, Object, Goal, and Long.
Sample 15 LIBERO+ variants per suite with seed 42 and run one trial per variant and checkpoint.
The setup downloads the three checkpoints from `VLADE-konkuk/checkpoint` using the Colab
secret `HF_TOKEN`. Google Drive supplies the evaluator source and stores the results.

The simulator uses the pinned LIBERO+ benchmark and simulation assets. The preprocessed
`VLADE-konkuk/libero-plus-splited` dataset is for training data splits. This rollout samples
from the benchmark registry and does not apply an RLDS split filter.

Run setup with `RUN_EVALUATION = False`, then set `RUN_EVALUATION = True` in a new cell
and rerun the final cell. The comparison JSON records results per suite and per model.

In [ ]:
from pathlib import Path, PurePosixPath
from google.colab import drive, userdata
from huggingface_hub import HfApi, snapshot_download
import datetime, hashlib, json, os, re, shutil, subprocess, tempfile, zipfile

RUN_EVALUATION = False
SEED = 42
TASK_SUITES = ('libero_spatial', 'libero_object', 'libero_goal', 'libero_10')
TRIALS_PER_TASK = 1
TASK_COUNT = 15
LIBERO_REV = '4976dc30028e805ff8094b55501d532c48fec182'
ASSETS_REV = 'dd2bd61b7d9a6fef1abc52d606e983b41886a149'
HF_REPO = 'VLADE-konkuk/checkpoint'
HF_REVISION = 'main'
# Optional exact repo-relative paths if the repository has duplicate checkpoint copies.
HF_CHECKPOINT_DIRS = {}

DRIVE = Path('/content/drive/MyDrive')
PROJECT = Path('/content/drive/.shortcut-targets-by-id/1jbXOZooUbe-SM9oUrKmNac_4Dfthkgs0/VLADE')
CODE = Path('/content/vlade_libero_plus_eval')
AVA = CODE / 'third_party/ava_vla'
SCRIPT = AVA / 'experiments/robot/libero/run_libero_eval.py'
LIBERO = CODE / 'LIBERO-plus'
VENV = CODE / 'py311'
PYTHON = VENV / 'bin/python'


def resolve_checkpoint_dirs(repo_files, overrides):
    folders = {}
    for filename in repo_files:
        path = PurePosixPath(filename)
        folders.setdefault(path.parent, set()).add(path.name)

    candidates = {
        folder: files for folder, files in folders.items()
        if 'config.json' in files
        and any(name.startswith('action_head') and name.endswith('.pt') for name in files)
    }
    aliases = {
        'T0': {'t0', 'avavlalibero4in1'},
        'T1-1975': {'t11975', 'step1975', 'step0001975'},
        'FFT-725': {'fft725', 'step725', 'step0000725'},
    }
    steps = {'T1-1975': '1975', 'FFT-725': '725'}
    selected = {}
    for label, names in aliases.items():
        if label in overrides:
            matches = [PurePosixPath(overrides[label])]
        else:
            matches = []
            for folder, files in candidates.items():
                parts = {re.sub(r'[^a-z0-9]', '', part.lower()) for part in folder.parts}
                step = steps.get(label)
                step_match = step and any(
                    re.fullmatch(rf'action_head--0*{step}_checkpoint\.pt', name)
                    for name in files
                )
                if names & parts or step_match:
                    matches.append(folder)
            if not matches and label == 'T1-1975':
                matches = [folder for folder in candidates if folder.name.lower() == 't1']
        if len(matches) != 1 or matches[0] not in candidates:
            raise RuntimeError(
                f'{label}: expected one checkpoint directory, found {sorted(map(str, matches))}. '
                f'Set HF_CHECKPOINT_DIRS[{label!r}] to its exact path. '
                f'Available checkpoint directories: {sorted(map(str, candidates))}'
            )
        selected[label] = matches[0]

    if len(set(selected.values())) != len(selected):
        raise RuntimeError(f'Expected three distinct checkpoint directories: {selected}')
    if 'COMPLETE' not in candidates[selected['FFT-725']]:
        raise RuntimeError(f"FFT-725: COMPLETE is missing in {selected['FFT-725']}")
    return {label: (folder, candidates[folder]) for label, folder in selected.items()}


def download_checkpoints():
    token = userdata.get('HF_TOKEN')
    api = HfApi(token=token)
    revision = api.repo_info(HF_REPO, repo_type='dataset', revision=HF_REVISION).sha
    if not revision:
        raise RuntimeError(f'Could not resolve the revision of {HF_REPO}')
    repo_files = api.list_repo_files(HF_REPO, repo_type='dataset', revision=revision)
    locations = resolve_checkpoint_dirs(repo_files, HF_CHECKPOINT_DIRS)
    checkpoints = {}
    sources = {}
    for label, (folder, files) in locations.items():
        filenames = sorted(str(folder / name) for name in files
                           if name not in {'trainer_state.pt', 'optimizer.pt'})
        snapshot = Path(snapshot_download(
            HF_REPO, repo_type='dataset', revision=revision, token=token,
            allow_patterns=filenames,
            local_dir=CODE / 'hf_checkpoints' / revision / label,
        ))
        checkpoint = snapshot.joinpath(*folder.parts)
        missing = [name for name in filenames if not (snapshot / name).is_file()]
        if missing:
            raise FileNotFoundError(f'{label}: missing checkpoint files: {missing}')
        checkpoints[label] = checkpoint
        sources[label] = {'repo': HF_REPO, 'revision': revision, 'path': str(folder)}
        print(label, sources[label], flush=True)
    return checkpoints, sources


CKPTS, HF_CHECKPOINT_SOURCES = download_checkpoints()
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_TOKEN"))
print(*api.list_repo_files(
    "VLADE-konkuk/checkpoint", repo_type="dataset", revision="main"
), sep="\n")

In [ ]:
AVA_INVENTORY = {'third_party/ava_vla/.pre-commit-config.yaml': '3f5526e4c92fcd179b8c7df3125d07272a9b4fe779b486052026aaa2daaa8f6b', 'third_party/ava_vla/LICENSE': '776b1de6987e63f473063601e3bf65633e81b4ed5ba2b5eebac6b5c5499c2e98', 'third_party/ava_vla/README.md': '3444db84c35a2882c5ea2f2ceb01f855de827416723a6a073b3766347bb893ee', 'third_party/ava_vla/experiments/robot/aloha/aloha_utils.py': 'e91291733b35f33539155989e087cbd24ed0aa1d81acf61cd65eb3d2f6dcb283', 'third_party/ava_vla/experiments/robot/aloha/constants.py': '3498c193544158b7b1156524a51af4886fc57d283d78ad54c6b2921d54779aa6', 'third_party/ava_vla/experiments/robot/aloha/preprocess_split_aloha_data.py': 'bb69c8a0453f0aef5969cfaca9bce11d4e8c5ef1fa5c3179ad55bc8dbf93058a', 'third_party/ava_vla/experiments/robot/aloha/real_env.py': '30006c4152bd0e22e982d4b4acda6062a71d59995e507915da90778457aa4631', 'third_party/ava_vla/experiments/robot/aloha/requirements_aloha.txt': '560784c894b0db4aed26299c436a8f31f70bb41c01646516ce5ef1c93df63f9c', 'third_party/ava_vla/experiments/robot/aloha/robot_utils.py': 'dc9a21730db88dba4ac922fc1e3700eb18bb8392c2f8a9d987b2fe5e3d1e4a72', 'third_party/ava_vla/experiments/robot/aloha/run_aloha_eval.py': 'c3e1929a311c603f781356ca13ff8c18ba9b300643b7097357f9431716d10e90', 'third_party/ava_vla/experiments/robot/calvin/__init__.py': '9b5448c1c0153d89e827224f4987f27a1f4609af43b5198e13946f705d9c4499', 'third_party/ava_vla/experiments/robot/calvin/calvin_env_wrapper.py': '4ff951474d5d0ef3bb1df7af14fc4147864a1733d441f3729ecf7896e4a290b2', 'third_party/ava_vla/experiments/robot/calvin/run_calvin_eval.py': 'de696ff04121e1b9f792526f2c3f18be598591381416a7580ca05c4cd5d15afa', 'third_party/ava_vla/experiments/robot/calvin/vla_evaluation.py': '8994476efecd971ffc40d63cbbe2b76cc59c9b709ad49a39c086d266946308de', 'third_party/ava_vla/experiments/robot/libero/libero_requirements.txt': '209837422576c29c62067b50cb6be9d0bb2b3ef6bd193f544612a192e477cbd3', 'third_party/ava_vla/experiments/robot/libero/libero_utils.py': '287d7d3663fa8f40d14ee9282e13e4a31065d250f436174dbab69d8288e8d708', 'third_party/ava_vla/experiments/robot/libero/regenerate_libero_dataset.py': '165fd2c08a6d29306de53b3ea9a0fbaf4a7e3d533d82464f21a38f2909ac0dd8', 'third_party/ava_vla/experiments/robot/libero/run_libero_eval.py': '3f2c36f5bda7035b7406510c1394e2207afeccb9d5829ddc04c88ff55d31e21f', 'third_party/ava_vla/experiments/robot/openvla_utils.py': 'c852bfd1ce8eddda21dc0c5b07a81ae8af7fa3325645ed52fc256d4f7a93789a', 'third_party/ava_vla/experiments/robot/robot_utils.py': '006234f1c097aeef51e553c865ce0c9d33d89f423c631ed51e3b26aee481f5f8', 'third_party/ava_vla/prismatic/__init__.py': '244d4c09ae7b191ae233484aed3a3dfb281bd1eb7af4e24392f047d0d42c6ef9', 'third_party/ava_vla/prismatic/conf/__init__.py': '2ce220a223ac5b351b6b9b3cd9ad8b16e880394e1b29dcd4b56753a7ddb72cbb', 'third_party/ava_vla/prismatic/conf/datasets.py': '0246b302e816c545738970cee2e881bcc6f7897593cce27c23d2b0c0baa4b1fa', 'third_party/ava_vla/prismatic/conf/models.py': '1f59b22a961c22ef0f1b514a4307d31707516f0617889aebefe852c6c0de4421', 'third_party/ava_vla/prismatic/conf/vla.py': '749a47163db615312b4daf847b987d157bc93cd62279e8fc83dff5a521b71069', 'third_party/ava_vla/prismatic/extern/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'third_party/ava_vla/prismatic/extern/hf/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'third_party/ava_vla/prismatic/extern/hf/configuration_prismatic.py': 'e13181aec9e6a789f876f51b9edbb65c58fccb0e85acb47304c7824938abfd4c', 'third_party/ava_vla/prismatic/extern/hf/modeling_prismatic.py': '9c1a50fa37a1e6843ae0eb1be016acc5d61e8d83b06b60b9ef0ce74ebc44ecb7', 'third_party/ava_vla/prismatic/extern/hf/processing_prismatic.py': '7c091d514ecde538fcc3491717dfe1e16296032530a2664bf472761a0cd9c579', 'third_party/ava_vla/prismatic/models/__init__.py': '5916dda5c2320fbab54f3ba09d67c8a52b2a5be7909d4c652caf0899d9f5eedb', 'third_party/ava_vla/prismatic/models/action_heads.py': 'c011bf5b2e0b05772c90081047122e50acc259923328810ff6c2fd148160034c', 'third_party/ava_vla/prismatic/models/attn_weight_generator.py': '939f1364618e47244c48fd032dbb645e7fc4aaa02a0f0a28e017a884e6b48c62', 'third_party/ava_vla/prismatic/models/backbones/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'third_party/ava_vla/prismatic/models/backbones/llm/__init__.py': '02791236db3febb7ff314310cd6dd8bac174bb7961118a3403995c60d8a95c09', 'third_party/ava_vla/prismatic/models/backbones/llm/base_llm.py': 'ef5b6e0d3d25e1ff177c82b58feae25ccd616f71c104d5fddf491fd4975bca6b', 'third_party/ava_vla/prismatic/models/backbones/llm/llama2.py': '8414694c9099da13831c0540dbc9bd851d172a28340ec4e39d18e2db264d7ba6', 'third_party/ava_vla/prismatic/models/backbones/llm/mistral.py': '4fc1aaaa966b9483bc1cbe98a0a98b4cd78b31b229c04fc97fdff1b082c03ac2', 'third_party/ava_vla/prismatic/models/backbones/llm/phi.py': 'a6d77b7e40b9175b0262910efcfa69347c11a9c94ab7207e523ce33afd90f330', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/__init__.py': '6f532ea1f274096e32a5b4bf9b98648f3e0521f1f7e30599487a3230663d7b92', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/base_prompter.py': '13640ed54b055c0fcd8daf7212be9f4faf76cec779fdd2e0f7ea5a866b0b6d51', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/llama2_chat_prompter.py': 'a35a82facfca67e202ad4ea3aaa778c72776b59eb8ee8936af8ce300ad49f2df', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/mistral_instruct_prompter.py': '3d5f57db4028ef84f8d09fb83e07f2049985c90a74cec471c764e353fe0fc617', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/phi_prompter.py': '7828ed56641f2251a5b0b45a2a6432834a24897854bed83434643a7008fc3b30', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/vicuna_v15_prompter.py': '06e1c31246300773973af7e5f0a36a2b2452653b81be1e983fbfaab5b506ae18', 'third_party/ava_vla/prismatic/models/backbones/vision/__init__.py': 'ed36c168ff2c4e230f7b31b3b68484aa48fb6700b58f14337e327126a00da2eb', 'third_party/ava_vla/prismatic/models/backbones/vision/base_vision.py': '4650adf0c2abe7bb23bb0010a0d00f3ffdc3585e298cba69a36c77cb3af97f5b', 'third_party/ava_vla/prismatic/models/backbones/vision/clip_vit.py': '2529c7a514aabe1d864654cb9783626e9252364cfb939c08f97c0db99f2021aa', 'third_party/ava_vla/prismatic/models/backbones/vision/dinoclip_vit.py': 'f4804c952dd7eb73bfad729934b11442f6f0294ddc1e1e37fa1e0b5d2bc293a0', 'third_party/ava_vla/prismatic/models/backbones/vision/dinosiglip_vit.py': '3799119de6d5d4f471eb841b30135e78efd8a8199b3c4af3db78f1a3a99fcfe3', 'third_party/ava_vla/prismatic/models/backbones/vision/dinov2_vit.py': 'd55c2d1639f4800f34618daecc7b84b794f72e55ba1579cb8385f5d914512a91', 'third_party/ava_vla/prismatic/models/backbones/vision/in1k_vit.py': '8644aabfab30e9fc3a8296d04ff69f6f2b9bd2ebb20143fb5c61e15b64e56fc9', 'third_party/ava_vla/prismatic/models/backbones/vision/siglip_vit.py': '7098e14baf02cbc0e814dc23a5abe31948529556fd7f5cd156ba73fdcce381b3', 'third_party/ava_vla/prismatic/models/film_vit_wrapper.py': '36d90b2717f210b9a0da8f80ba0d33edb0916ef75745caa20d8f6f266a92da3f', 'third_party/ava_vla/prismatic/models/load.py': '2f9cc4535e5fc190669a69463b55cdc35709ce6f990055ecd9e20a0997e88604', 'third_party/ava_vla/prismatic/models/materialize.py': 'cd7aecf09de31b7fb18ac2795bf7b5f97f9e07157dbfd5d588e6414d9904d94a', 'third_party/ava_vla/prismatic/models/projectors.py': 'ca0fc18af5ba38501a9642ac6da4a64047e2ed013cd976ea3da2632ca33eb2e1', 'third_party/ava_vla/prismatic/models/registry.py': '4e226a6f6182ec4ecbdfd8507c85831cbc43de03a5cbb715af712092a7482b21', 'third_party/ava_vla/prismatic/models/vlas/__init__.py': 'b55e834c024738ce1ca7f69392c01262f19078343c57f264177fe347a694dfe0', 'third_party/ava_vla/prismatic/models/vlas/openvla.py': 'c364351dc787ad7a6fb5ee6168532a7c4ba6d0a2c98b65836f20708524d71337', 'third_party/ava_vla/prismatic/models/vlms/__init__.py': '0f5b26ba8114b3f1717aa693568bfb3dcf52ab7b2f2b6d7fe0bd69ff19751817', 'third_party/ava_vla/prismatic/models/vlms/base_vlm.py': '1c0c8815d745bc01f8c6d93b893aded94cdc55fe6ffcbdf12a255a8784b3921f', 'third_party/ava_vla/prismatic/models/vlms/prismatic.py': '612b4369e77880c3fd3ae7960fdf136b745d6a2dce57176d12160f30d88de20f', 'third_party/ava_vla/prismatic/overwatch/__init__.py': '3e9c13fa52c707586621c75406253086340c4667904d0053e3f34ee991b2241a', 'third_party/ava_vla/prismatic/overwatch/overwatch.py': '7e065cd22a6337a1b9a43fb4f456d7fed1231345b147f7a2a0f48cfc118af45d', 'third_party/ava_vla/prismatic/preprocessing/__init__.py': '5681e8f6c09e4d86198b8107b9d7251c90b5b455b04080827d28f37d8bf56372', 'third_party/ava_vla/prismatic/preprocessing/datasets/__init__.py': '72e19b90d709ebd6178bf879e5f9dfecbc83631642c8d31c264c955733cb4ef1', 'third_party/ava_vla/prismatic/preprocessing/datasets/datasets.py': 'e51471c98f85fba6f4e2fe03204bbbf2a176163931e700dde2310affb5832575', 'third_party/ava_vla/prismatic/preprocessing/download.py': '257fda53e11217f3c3cd0b9092c8064fe8a7af5910edf9bd0cb8f7f1d29c1b9d', 'third_party/ava_vla/prismatic/preprocessing/materialize.py': 'f98ab5e3ba03b6cbe348e8a488a41ece3633cb7c46fd82836a20f10484938772', 'third_party/ava_vla/prismatic/training/__init__.py': '625226108c7459cc0fed8cb26193496980648c1060d2ff42a8a9400dd3bf51d6', 'third_party/ava_vla/prismatic/training/materialize.py': '80d693f2144989ac5687fb243b4ca9a6648e25df10c83cf493f0010b5f657b12', 'third_party/ava_vla/prismatic/training/metrics.py': 'd0ee5dee7f7141a4be5b4622580abe6f1b6d183cb995abcbcf1cbb120d1a6fc2', 'third_party/ava_vla/prismatic/training/strategies/__init__.py': '7d65535181ee4fd105dae95e69e5de1a7599a9f31bb8f97314149c62807ad06d', 'third_party/ava_vla/prismatic/training/strategies/base_strategy.py': '5fb67f5d867af89e559c1f3e64292c8018285c10950d8b6baf078cea5f728158', 'third_party/ava_vla/prismatic/training/strategies/ddp.py': 'db038a951ad1d8d0ffd70f8e4d8b58d89c85e209f7a8aaf1ad4eb408266d0138', 'third_party/ava_vla/prismatic/training/strategies/fsdp.py': 'ca1b90934237e604ef02f017f79ea7eed3d5d9c1431e9512f92a2493e4fcc52d', 'third_party/ava_vla/prismatic/training/train_utils.py': '6ecc95436fde63ce023e2ba7151826d96a915b5d5c2ec5698750022580b0e7b3', 'third_party/ava_vla/prismatic/util/__init__.py': '8ab7190ed1f1bd087966fcc59f35f9a7b4b315dd7b17f68a6a7ed6e335fc7380', 'third_party/ava_vla/prismatic/util/batching_utils.py': 'a41d58f7baa40ccefe92be2727c19963a720a41feeada301a09cf01c8b261feb', 'third_party/ava_vla/prismatic/util/data_utils.py': '431ef4210296c2165142b1226a4adad985d99d0391cd822f1630e98fbb6c9d7d', 'third_party/ava_vla/prismatic/util/nn_utils.py': '7aa0562eab374f0585bb53639b861f06b6c44b82384e938dda9cd32d844bc0d0', 'third_party/ava_vla/prismatic/util/temporal_context_utils.py': '169a74905a36015f00a921558cf76e27cc18f502daf20beddab32a5c55032da2', 'third_party/ava_vla/prismatic/util/torch_utils.py': 'a70e3b8d7e24b15a5d8abfa94f1c596ea70a9a85c86b50e8d8f963af51ff3304', 'third_party/ava_vla/prismatic/vla/__init__.py': 'bfc31eb7e16374695cc0da879bb1e43451395ee64144da896f00e2187862f069', 'third_party/ava_vla/prismatic/vla/action_tokenizer.py': 'fdc98fcbf5b0926ef2181db71946d23ffbfa052cf8443dc933d52c42a191352c', 'third_party/ava_vla/prismatic/vla/constants.py': '5b53c3a07cf1ed27bf94a88da9929e82818792d6fdc32343d133afea1a0acf3d', 'third_party/ava_vla/prismatic/vla/datasets/__init__.py': 'f40c0fa32bbb2f52d9a8a4cd1acfa061b4e0da0e0e6e4dc290ae6f351cf62f04', 'third_party/ava_vla/prismatic/vla/datasets/datasets.py': '36254f1112e66af6c1394ebf8279fc505bb327306ce892c78f321831b34ca739', 'third_party/ava_vla/prismatic/vla/datasets/rlds/__init__.py': '35b3206143eaf348bdb05401dddc4c14b382761be2832d8f64a8e6bf8e4c404c', 'third_party/ava_vla/prismatic/vla/datasets/rlds/dataset.py': '4acfd0ebfe7e2765de1a4e5fd19ceb0939cd77ef1b022756f47871ccdae260f1', 'third_party/ava_vla/prismatic/vla/datasets/rlds/obs_transforms.py': '1f2eddd9e2fe5ea96be294945567e665172287973be5753a7381d777c7d40392', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/__init__.py': '4a8a96f8c2af6c5b56f2bc29493141c160dcd3595ca977a95245b2ba64598157', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/configs.py': '0995257a400c5d396cb75123cd3567283d701ba89a037dc7e0aecba7e2dcbf9b', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/materialize.py': '375c8117e02fec5ff5971d1d9c1fb021ec2e615aa357be21bbb3d5e5d5001496', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/mixtures.py': '0598aac5251f44d70d5e0c609977d32afcda30c72098329005d8559bb35786bd', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/transforms.py': '7644f1b4fa0badab70f9924de68eeb9d94b41c82bf6d8dbc776e58677adae485', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/utils/droid_utils.py': '6eddeaf36eb335656435f8773ac715ab966994c2b97cf72f257751140c3c4099', 'third_party/ava_vla/prismatic/vla/datasets/rlds/traj_transforms.py': '1ade02497222b16f74da1a005a2d22829d109e3c1855c14b078dc5cbdb799566', 'third_party/ava_vla/prismatic/vla/datasets/rlds/utils/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'third_party/ava_vla/prismatic/vla/datasets/rlds/utils/data_utils.py': '4f7a36c6713e6c28a7237ec7ae1325abd47a1f0c921164fc5548ce4c81340d44', 'third_party/ava_vla/prismatic/vla/datasets/rlds/utils/goal_relabeling.py': 'e59594065d9497ca4a3e51fe53d6687fbd6990a966d67a36528524e7cf22d360', 'third_party/ava_vla/prismatic/vla/datasets/rlds/utils/task_augmentation.py': '1593b89224e6d70552b13f182e42194ea348b5dd1ea9b0507b4fa09642f25113', 'third_party/ava_vla/prismatic/vla/materialize.py': 'dbc1b0c88e3acb926829e7d86f00b7aef52b6679af641f5ad328aa59d99f7ab5', 'third_party/ava_vla/pyproject.toml': '906f17ee278cef3bb4352f0dc80967986d8686151d3905856fa69f673bbb9f28', 'third_party/ava_vla/scripts/extern/convert_prismatic_weights_to_hf.py': '75d9921fa41d4c3a5540aa478d8057da47fb4dc324c74737557cb0abd76f23a2', 'third_party/ava_vla/scripts/extern/verify_prismatic.py': 'a0d13b4960997f8d09667139ef170c3dd9257871b44e04cc0d3437dba55bb783', 'third_party/ava_vla/vla-scripts/deploy.py': '4d00d69b594d694181e05d6da7e675b5a1d57c72c310a7e8706700c49f16aca5', 'third_party/ava_vla/vla-scripts/extern/convert_openvla_weights_to_hf.py': 'f1ad6f8db1e5c52a5fc39d8f4f487c62cd6308e2469805358efbb02f4c505809', 'third_party/ava_vla/vla-scripts/extern/verify_openvla.py': 'baee65706be73d70a8b163b223ae9f4a91141c21edafd5f7ee0976b3ff7ad47f', 'third_party/ava_vla/vla-scripts/finetune.py': '1400729cd444559dd914f9c3551868708592c79cbab96462cf80ad8c2bb0699c', 'third_party/ava_vla/vla-scripts/merge_lora_weights_and_save.py': '84bd8d83a511e7e98e9160c1b69d3508ac8e40586f98cc3b9f6b9c781733a7f1'}
for rel, expected in AVA_INVENTORY.items():
    dst = CODE / rel
    if dst.is_file() and hashlib.sha256(dst.read_bytes()).hexdigest() == expected:
        continue
    payload = (PROJECT / rel).read_bytes()
    if hashlib.sha256(payload).hexdigest() != expected:
        raise ValueError(f'AVA source SHA mismatch: {rel}')
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_bytes(payload)

In [ ]:
import shutil, subprocess, os

def run_cmd(cmd, cwd=None, env=None):
    print('RUN:', ' '.join(map(str,cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, env=env, check=True)

run_cmd(['apt-get','update','-qq'])
run_cmd(['apt-get','install','-y','python3.11','python3.11-venv','python3.11-dev',
         'build-essential','linux-libc-dev'])
if not PYTHON.is_file():
    run_cmd(['python3.11','-m','venv',str(VENV)])
run_cmd([PYTHON,'-m','pip','install','-U','pip','setuptools','wheel'])
# AVA 원본 pyproject의 torch2.2/cu121 (기존 평가 노트북과 동일)
run_cmd([PYTHON,'-m','pip','install','torch==2.2.0','torchvision==0.17.0','torchaudio==2.2.0',
         '--index-url','https://download.pytorch.org/whl/cu121'])
run_cmd([PYTHON,'-m','pip','install','-e',str(AVA)])
# LIBERO: 설치 이전에 Linux 빌드 패키지를 준비해 evdev wheel 오류 차단
run_cmd([PYTHON,'-m','pip','install','evdev'])
run_cmd([PYTHON,'-m','pip','install',
         'robosuite==1.4.1','bddl','future','easydict==1.9','gym==0.25.2',
         'termcolor','h5py','tqdm','PyYAML','imageio',
         'numpy==1.26.4','opencv-python<4.12',
         'tensorflow-metadata==1.15.0'])
# 이전 rollout 노트북 18번 셀의 명시 버전 (2.3.7 vs 3.3.1 혼합 금지)
run_cmd([PYTHON,'-m','pip','install','--no-deps','mujoco==3.3.1'])

if not LIBERO.exists():
    run_cmd(['git', 'clone', 'https://github.com/sylvestf/LIBERO-plus.git', LIBERO])
run_cmd(['git', '-C', LIBERO, 'checkout', '--detach', LIBERO_REV])
actual_revision = subprocess.check_output(['git', '-C', str(LIBERO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_revision == LIBERO_REV
run_cmd([PYTHON, '-m', 'pip', 'install', '--no-deps', '-e', LIBERO])
assets_root = LIBERO / 'libero/libero'
receipt = assets_root / '.vlade_assets_revision'
if not receipt.is_file() or receipt.read_text().strip() != ASSETS_REV:
    archive = PROJECT / 'data/raw/libero_plus_assets/assets.zip'
    if not archive.is_file():
        archive = Path(subprocess.check_output([
            str(PYTHON), '-c',
            'from huggingface_hub import hf_hub_download; import sys; print(hf_hub_download("Sylvest/LIBERO-plus","assets.zip",repo_type="dataset",revision=sys.argv[1]))',
            ASSETS_REV], text=True).strip())
    with zipfile.ZipFile(archive) as files:
        for name in files.namelist():
            relative = Path(name)
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError(f'Invalid assets archive member: {name}')
        files.extractall(assets_root)
    if not (assets_root / 'assets' / 'textures').is_dir():
        raise RuntimeError('LIBERO+ asset extraction incomplete')
    receipt.write_text(ASSETS_REV)

In [ ]:
ENV = os.environ.copy()
ENV.update({
    'MPLBACKEND':'Agg','MUJOCO_GL':'egl','PYOPENGL_PLATFORM':'egl',
    'PYTHONPATH': f'{AVA}:{CODE / "src"}:{LIBERO}:' + ENV.get('PYTHONPATH',''),
    'TOKENIZERS_PARALLELISM':'false','WANDB_MODE':'disabled',
})
sim_program = r"""
import sys
import torch, numpy, mujoco, robosuite
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv
print('versions:', 'torch',torch.__version__,'numpy',numpy.__version__,
      'mujoco',mujoco.__version__,'robosuite',robosuite.__version__)
assert torch.cuda.is_available(), 'CUDA not available'
print('GPU:',torch.cuda.get_device_name(0))
expected_counts = {
    'libero_spatial': 2402, 'libero_object': 2518, 'libero_goal': 2591, 'libero_10': 2519,
}
registry = benchmark.get_benchmark_dict()
for name in sys.argv[1:]:
    suite = registry[name]()
    if suite.n_tasks != expected_counts[name]:
        raise RuntimeError(f'{name}: expected {expected_counts[name]} LIBERO+ variants, got {suite.n_tasks}')
    print(name, suite.n_tasks)
suite = registry[sys.argv[1]]()
sim=OffScreenRenderEnv(bddl_file_name=suite.get_task_bddl_file_path(0),camera_heights=128,camera_widths=128)
try:
    obs=sim.reset()
    assert isinstance(obs,dict) and len(obs)>0
    print('LIBERO PLUS SIMULATION PASS',len(obs))
finally:
    sim.close()
"""
# Always allow first-run configuration prompt through stdin; harmless once initialized.
r = subprocess.run([str(PYTHON),'-c',sim_program,*TASK_SUITES],input='N\n', cwd=str(LIBERO),
                   env=ENV,text=True,capture_output=True,timeout=240)
print('Return code:',r.returncode,'\nSTDOUT:\n',r.stdout[-5000:],'\nSTDERR:\n',r.stderr[-5000:])
assert r.returncode==0 and 'LIBERO PLUS SIMULATION PASS' in r.stdout, 'SIMULATION FAILED: stop here'
SIMULATION_READY=True

In [ ]:
EVAL_CKPTS = {}
VIEW_ROOT = Path(tempfile.mkdtemp(prefix='ava_eval_views_', dir=CODE))
for label, src_path in CKPTS.items():
    dst = VIEW_ROOT / label
    dst.mkdir()
    for name in ('action_head', 'proprio_projector', 'vision_attn_weight_generator'):
        matches = list(src_path.glob(f'{name}*.pt'))
        if len(matches) != 1:
            raise RuntimeError(f'{label}: expected one {name} checkpoint, found {matches}')
    for file in src_path.iterdir():
        if not file.is_file() or file.name in {'trainer_state.pt', 'optimizer.pt'}:
            continue
        target = dst / file.name
        if file.suffix in {'.json', '.py'}:
            shutil.copy2(file, target)
        else:
            target.symlink_to(file)
    if label != 'T0':
        config_path = dst / 'config.json'
        stats_path = dst / 'dataset_statistics.json'
        config = json.loads(config_path.read_text())
        stats = json.loads(stats_path.read_text())
        fft_stats = stats['libero_plus_mixdata']
        suite_stats = dict.fromkeys(TASK_SUITES, fft_stats)
        stats.update(suite_stats)
        config.setdefault('norm_stats', {}).update({'libero_plus_mixdata': fft_stats, **suite_stats})
        stats_path.write_text(json.dumps(stats, indent=2))
        config_path.write_text(json.dumps(config, indent=2))
    EVAL_CKPTS[label] = dst

In [ ]:
EVAL_RUNNER = """
import importlib.util
import inspect
import json
import random
import sys
from pathlib import Path

script, result_path, suite_name, seed, count, trials = sys.argv[1:7]
sys.argv = [script, *sys.argv[7:]]
sys.path.insert(0, str(Path(script).parent))
spec = importlib.util.spec_from_file_location('vlade_eval_entry', script)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)
suite = module.benchmark.get_benchmark_dict()[suite_name]()
task_ids = sorted(random.Random(int(seed)).sample(range(suite.n_tasks), int(count)))
task_set = set(task_ids)
run_task = module.run_task
signature = inspect.signature(run_task)
log_message = module.log_message
records = []
def fail_on_episode_error(message, *args, **kwargs):
    log_message(message, *args, **kwargs)
    if str(message).startswith('Episode error:'):
        raise RuntimeError(message)
def selected_task(*args, **kwargs):
    inputs = signature.bind(*args, **kwargs).arguments
    task_id = inputs['task_id']
    before = inputs['total_episodes'], inputs['total_successes']
    if task_id not in task_set:
        return before
    after = run_task(*args, **kwargs)
    rollouts, successes = after[0] - before[0], after[1] - before[1]
    if rollouts != int(trials) or not 0 <= successes <= rollouts:
        raise RuntimeError(f'Expected {trials} valid rollouts for task {task_id}')
    records.append({'task_id': task_id, 'successes': successes, 'rollouts': rollouts})
    Path(result_path).write_text(json.dumps(records, indent=2))
    return after
module.log_message = fail_on_episode_error
module.run_task = selected_task
module.eval_libero()
if [row['task_id'] for row in records] != task_ids:
    raise RuntimeError('Incomplete or reordered task results')
"""

if RUN_EVALUATION:
    assert SIMULATION_READY
    run_id = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    out = DRIVE / 'VLADE/evaluations/libero_plus_ava_paired' / run_id
    out.mkdir(parents=True, exist_ok=False)
    summary_path = out / 'comparison.json'
    comparison = {
        'seed': SEED, 'tasks_per_suite': TASK_COUNT, 'trials_per_task': TRIALS_PER_TASK,
        'libero_plus_revision': LIBERO_REV, 'assets_revision': ASSETS_REV,
        'checkpoints': HF_CHECKPOINT_SOURCES, 'suites': {}, 'models': {},
        'complete': False,
    }
    for suite_name in TASK_SUITES:
        suite_comparison = {'task_ids': None, 'models': {}}
        comparison['suites'][suite_name] = suite_comparison
        for label, selected in EVAL_CKPTS.items():
            folder = out / suite_name / label
            folder.mkdir(parents=True)
            result_path = folder / 'task_results.json'
            args=[
                '--model_family','openvla',
                '--pretrained_checkpoint',str(selected),
                '--use_l1_regression','True','--use_diffusion','False','--use_proprio','True',
                '--num_images_in_input','2','--num_open_loop_steps','8',
                '--task_suite_name',suite_name,'--num_trials_per_task',str(TRIALS_PER_TASK),
                '--seed',str(SEED),'--use_wandb','False',
                '--multi_frame.temporal_strategy','attn_weight',
                '--multi_frame.temporal_feature_source','action',
                '--multi_frame.temporal_feature_layer','-2',
                '--multi_frame.attn_weight_force_eager_attn','True',
                '--multi_frame.attn_weight_extra_layer_ids','(i for i in range(0, 32))',
                '--multi_frame.attn_weight_head_type','softmaxscore',
                '--multi_frame.attn_weight_score_config','(1.9, 0.1, 0.0)',
                '--multi_frame.attn_weight_sink_ids','[68,75,180,187,324,331,436,443]',
                '--multi_frame.attn_weight_sink_weight','1.0',
            ]


            args += ['--local_log_dir', str(folder)]
            (folder / 'run.json').write_text(json.dumps({
                'checkpoint': HF_CHECKPOINT_SOURCES[label], 'suite': suite_name,
                'libero_plus_revision': LIBERO_REV, 'assets_revision': ASSETS_REV,
                'seed': SEED, 'args': args,
            }, indent=2))
            command = [str(PYTHON), '-c', EVAL_RUNNER, str(SCRIPT), str(result_path),
                       suite_name, str(SEED), str(TASK_COUNT), str(TRIALS_PER_TASK), *args]
            log_path = folder / 'console.log'
            with log_path.open('w', encoding='utf-8') as log:
                result = subprocess.run(command, cwd=AVA, env=ENV, stdout=log, stderr=subprocess.STDOUT)
            if result.returncode:
                raise RuntimeError(f'{suite_name}/{label} evaluation failed. See {log_path}')
            rows = json.loads(result_path.read_text())
            tasks = [row['task_id'] for row in rows]
            if suite_comparison['task_ids'] is None:
                suite_comparison['task_ids'] = tasks
            if tasks != suite_comparison['task_ids'] or len(tasks) != TASK_COUNT:
                raise RuntimeError(f'{suite_name}/{label}: paired task IDs do not match')
            successes = sum(row['successes'] for row in rows)
            rollouts = sum(row['rollouts'] for row in rows)
            suite_comparison['models'][label] = {
                'successes': successes, 'rollouts': rollouts, 'success_rate': successes / rollouts,
            }
            summary_path.write_text(json.dumps(comparison, indent=2))
            print(suite_name, label, suite_comparison['models'][label], flush=True)

    for label in EVAL_CKPTS:
        model_suites = [suite['models'][label] for suite in comparison['suites'].values()]
        successes = sum(suite['successes'] for suite in model_suites)
        rollouts = sum(suite['rollouts'] for suite in model_suites)
        comparison['models'][label] = {
            'successes': successes, 'rollouts': rollouts, 'success_rate': successes / rollouts,
        }
    comparison['complete'] = True
    summary_path.write_text(json.dumps(comparison, indent=2))
    print(out)
else:
    print('Setup mode. Set RUN_EVALUATION=True in this runtime before running this cell again.')